# Stage 6 — Cross-Symptom Routing

For each admission, looks at every pair of Stage 5's *already-grounded* `CURRENT_SYMPTOMS`
concepts and asks: how closely related are they in the SNOMED is-a hierarchy? If two
independently-extracted symptoms turn out to share a deep, specific common ancestor, that
convergence is structural evidence pointing toward a candidate diagnosis — without relying on
UMLS's sparse curated relation labels (`finding_of`, `manifestation_of`) the old, abandoned
Stage 6 design leaned on.

**Input** : `patient_records/<patient>/admissions/<hadm>/stage_05_ontology_routing_agent/routed_terms.json`
**Output**: `patient_records/<patient>/admissions/<hadm>/stage_06_cross_symptom_routing/symptom_relations.json`

## Why shared-ancestor-with-distance, not exact-concept-match

An earlier version tried "do two symptoms' neighborhoods contain the exact same concept?" — a
quick scan across all 15 real admissions found that fires meaningfully in only 1 of them; the
rest showed either zero overlap or a degenerate one (two near-duplicate extractions of the same
symptom, e.g. "cough" vs "coughing"). Too sparse to be useful.

Wu-Palmer similarity already measures exactly what "how distant is the shared ancestor" means —
its formula, `2×depth(LCS)/(depth(A)+depth(B))`, is high when the Lowest Common Subsumer (LCS)
is deep/specific, and low when it's shallow/generic (or, in the extreme, when the only shared
ancestor is the absolute SNOMED root, giving a score of exactly 0.0 — meaning the two concepts
live in unrelated branches of the ontology entirely).

## A real limitation this ran into, and the fix: attribute sharing

Wu-Palmer measures *taxonomic* distance — where a concept sits in the is-a tree — which is not
the same as *causal/clinical* relatedness. On real data: a patient with multifocal metastatic
HCC (liver cancer) and hepatic encephalopathy scored **0.0000** between those two concepts, even
though HCC very plausibly caused the liver failure that caused the encephalopathy in this
patient. The reason: HCC lives under SNOMED's neoplasm/tumor branch, hepatic encephalopathy
lives under metabolic/neurological disorders — entirely different branches, so is-a distance
sees them as unrelated even though they're clinically connected.

SNOMED concepts also carry *defining attributes*, separate from the is-a hierarchy, that
capture *why* a concept is what it is: `has_finding_site` (anatomical structure),
`has_causative_agent` (organism/substance responsible), and `has_associated_morphology` (the
type of pathological process). Two concepts in unrelated is-a branches can still share one of
these — and in fact HCC and Hepatic encephalopathy **do** share the exact same `finding_site`
(Liver structure), a connection Wu-Palmer alone completely missed. Attribute sharing is
reported as its **own separate signal** alongside the Wu-Palmer score, not blended into it —
same philosophy as keeping cosine and Wu-Palmer as separate columns elsewhere in this project,
rather than opaquely combining metrics that measure genuinely different things.

## How the big-gap heuristic applies here

Each admission's pairwise Wu-Palmer scores are sorted descending and run through the same
median/MAD-based `biggest_gap_cutoff()` already used in Stage 5 — reused as-is, not
reimplemented. This determines `significant_connection` for each pair. Attribute-sharing is
*not* run through its own cutoff — it's a much sparser, more binary signal (most pairs share
nothing), so it's reported as a plain score alongside each pair rather than statistically
tested the same way.

## Scope

Only `CURRENT_SYMPTOMS` terms that Stage 5 successfully grounded are considered — ungrounded
terms and `DOCUMENTED_DIAGNOSES`/prior history are still excluded, same reasoning as Stage 5:
this is about finding structural evidence for *inferring* a diagnosis, not re-deriving something
already known.

**Update -- active metric: cosine similarity**. Two graph-structural alternatives were tried and tested against real admissions and are kept fully implemented but commented out below, both for the same reason: reusing `biggest_gap_cutoff()` unchanged across metrics it wasn't calibrated for produced misleading results. Wu-Palmer collapses the whole shared lineage to a single deepest-ancestor ratio, and its scores (clustered around 0.55-0.66 on this data) work reasonably with the existing cutoff. Ancestor-set Jaccard similarity (`|A∩B| / |A∪B|`, considering the whole shared lineage rather than one ancestor) produced a much more compressed, tie-heavy score range -- real ties (e.g. three pairs all scoring exactly 0.2727) collapsed the cutoff's median-absolute-deviation statistic to 0 and produced meaningless multi-million z-scores, on top of the cutoff only ever returning a single split point even when multiple genuine breaks existed further down the ranked list. Both are real, fixable issues, just not resolved yet -- see `ancestor_jaccard_with_lcs()` and `wu_palmer_with_lcs()` in `snomed_ontology.py`, and the commented-out blocks in the cell below.

**Noted for future work, not implemented here**: (1) recalibrating `biggest_gap_cutoff()` for Jaccard's scale (excluding zero/near-zero gaps from the median/MAD calculation, and taking the deepest qualifying outlier rather than only the largest-valued one) -- note `biggest_gap_cutoff()` is shared with Stage 5, so this would need either a Jaccard-specific variant or a decision to change it globally; (2) re-testing Wu-Palmer on this data now that the duplicate-concept exclusion and grounding fixes are in place; (3) multi-hop causal-chain traversal over SNOMED's `due_to`/`cause_of` relations (e.g. HCC → due_to → liver failure → due_to → hepatic encephalopathy) -- a structurally different signal from either is-a distance or text similarity, closer to genuine causal/mechanistic convergence, not yet built.

## 1. Setup

In [1]:
import sys
import json
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))

from snomed_ontology import (
    configure,
    load_umls_api_key,
    ancestor_jaccard_with_lcs,  # kept for the commented-out Jaccard block below
    wu_palmer_with_lcs,         # kept for the commented-out Wu-Palmer block below
    shared_attributes,
    direct_causal_link,
    get_concept_name,
    biggest_gap_cutoff,
    embed,
    cosine_sim,
)

PROJECT_ROOT = NB_DIR.parent
RECORDS_DIR       = PROJECT_ROOT / "patient_records"
STAGE_05_OUTPUT   = "stage_05_ontology_routing_agent"
STAGE_06_OUTPUT   = "stage_06_cross_symptom_routing"

UMLS_API_KEY = load_umls_api_key(PROJECT_ROOT)
configure(UMLS_API_KEY)
print(f"UMLS key loaded: {bool(UMLS_API_KEY)}")

patients = sorted([p for p in RECORDS_DIR.iterdir() if p.is_dir() and p.name.startswith("patient_")])
print(f"Patients found: {len(patients)}")


UMLS key loaded: True
Patients found: 15


## 2. Attribute-sharing weights — ADJUST HERE

These five constants are the only thing you should need to change to explore different
weightings — everything else in this notebook reads from them, so editing these and re-running
from here down (no need to touch `compute_symptom_relations` itself) is enough to test a new
combination.

**Recommended starting ranges** (reasoning: how specific/rare a coincidental match would be):

| Attribute | Suggested range | Why |
|---|---|---|
| `WEIGHT_CAUSE_OF` | 0.35 – 0.55 | Strongest — a *direct* causal assertion between these two specific concepts (A causes B, or vice versa), not just a shared trait. Weighted at/above causative_agent since it's a more specific claim. |
| `WEIGHT_CAUSATIVE_AGENT` | 0.30 – 0.50 | Sharing the exact organism/substance responsible is rarely coincidental |
| `WEIGHT_FINDING_SITE` | 0.15 – 0.30 | Meaningful but coarser — many unrelated conditions share the same organ |
| `WEIGHT_ASSOCIATED_MORPHOLOGY` | 0.10 – 0.20 | Weakest alone — many concepts share generic morphologies |
| `WEIGHT_PATHOLOGICAL_PROCESS` | 0.10 – 0.20 | Same tier as morphology — a generic process category (e.g. "Infectious process") shared by many unrelated concepts |

These ranges are a reasoned starting point, not a validated answer — there's no ground-truth
label set yet to confirm any specific number against (see the grid-search discussion below).
Set any weight to `0.0` to disable that signal entirely.

**Why `cause_of` isn't scored the same way as the other four**: `has_finding_site`,
`has_causative_agent`, `has_associated_morphology`, and `has_pathological_process` all ask "do
concepts A and B reference the *same third-party value*?" (e.g., both affecting Liver
structure). `cause_of` asks something different: "does A *directly* cause B, or B cause A?" —
a direct assertion between this specific pair, not a shared attribute. That's why it's computed
via `direct_causal_link()` rather than `shared_attributes()`.

**What else was considered and deliberately left out**: a full scan of SNOMED's relation types
on real concepts turned up several others — `mapped_to` (cross-vocabulary code mapping, purely
administrative), `possibly_equivalent_to`/`same_as`/`replaces` (terminology maintenance —
deprecated/merged concepts), `referred_to_by`/`refers_to` (lexical synonyms, not clinical
linkage), `isa`/`inverse_isa` (this *is* the is-a hierarchy, already measured by Wu-Palmer —
including it here would double-count it), and `focus_of`/`associated_finding_of`
(administrative/history-tracking, weak or off-topic). None of these represent genuine
clinical relatedness beyond what's already captured, so they're excluded rather than silently
zero-weighted alongside signals that do matter.


In [2]:
WEIGHT_CAUSE_OF = 0.45
WEIGHT_CAUSATIVE_AGENT = 0.40
WEIGHT_FINDING_SITE = 0.25
WEIGHT_ASSOCIATED_MORPHOLOGY = 0.15
WEIGHT_PATHOLOGICAL_PROCESS = 0.15


def compute_attribute_score(sctid_a: str, sctid_b: str):
    """Check whether two concepts share any defining attribute value, or have a
    direct causal link. Returns (attribute_score, shared_details) --
    shared_details names which specific concept was shared per attribute (or
    True/False for cause_of, since that's a direct link, not a shared value),
    for transparency, same idea as showing the LCS name alongside the
    Wu-Palmer score.

    cause_of is checked separately from the other four (see direct_causal_link's
    docstring) -- it's "does A directly cause B (or vice versa)", not "do A and
    B reference the same third-party value"."""
    shared = shared_attributes(sctid_a, sctid_b)
    weights = {
        "causative_agent": WEIGHT_CAUSATIVE_AGENT,
        "finding_site": WEIGHT_FINDING_SITE,
        "associated_morphology": WEIGHT_ASSOCIATED_MORPHOLOGY,
        "pathological_process": WEIGHT_PATHOLOGICAL_PROCESS,
    }
    attribute_score = sum(weights[attr] for attr, sctid in shared.items() if sctid is not None)
    shared_details = {attr: (get_concept_name(sctid) if sctid else None) for attr, sctid in shared.items()}

    has_causal_link = direct_causal_link(sctid_a, sctid_b)
    if has_causal_link:
        attribute_score += WEIGHT_CAUSE_OF
    shared_details["cause_of"] = has_causal_link

    return round(attribute_score, 4), shared_details


# Sanity check against the real case that motivated this: HCC vs Hepatic encephalopathy,
# unrelated by Wu-Palmer (0.0) but should now show a shared finding_site.
print(compute_attribute_score("109841003", "59927004"))

# Sanity check for the new cause_of signal: Hepatic encephalopathy directly causes
# "Dementia due to hepatic failure" -- should show cause_of: True.
print(compute_attribute_score("59927004", "1259465009"))


(0.25, {'finding_site': 'Liver structure', 'causative_agent': None, 'associated_morphology': None, 'pathological_process': None, 'cause_of': False})
(0.45, {'finding_site': None, 'causative_agent': None, 'associated_morphology': None, 'pathological_process': None, 'cause_of': True})


## On grid search — is it worth it?

Short answer: **not yet**, for a specific reason rather than "it's not possible."

Grid search needs an objective function to score each weight combination against — some
ground truth telling you "this combination of weights produced better results than that one."
Right now there isn't one: there's no labeled set of "these symptom pairs should be flagged
related, these shouldn't" to optimize against, and the actual downstream consumer of this score
(a future ICD-10 prediction/confidence stage) doesn't exist yet either. Without that, a grid
search would just explore the parameter space with no way to tell which point is actually
better — you'd still have to manually eyeball the results to judge that, which is exactly what
manually adjusting the three constants above already gets you, without the extra machinery.

There's also an overfitting risk specific to this dataset: only 15 admissions, most with a
handful of grounded symptoms. Tuning weights to whatever looks best on this small sample risks
finding weights that fit these 15 patients' idiosyncrasies rather than anything that
generalizes.

**When it would become worth it**: once Stage 7 (final ICD-10 decision) exists and there's a
way to measure prediction quality against ground truth (precision/recall against the real ICD
codes already sitting in each patient's `ground_truth.txt`/`admission_history.json`, the same
evaluation pattern the old abandoned Stage 6/7 notebooks used) — at that point weight tuning has
an actual objective to optimize against, and a grid search (or a more sample-efficient method,
since it's only 3 parameters) becomes a reasonable thing to automate. Until then, manual
exploration via the constants above is the right-sized approach.


## 3. Compute pairwise relations for one admission

Pulls the grounded `{term: sctid}` map out of Stage 5's output, computes a pairwise
similarity score *and* the attribute-sharing score for every pair, sorts descending by
that score, and applies the big-gap cutoff to flag which pairs are statistically
significant connections. Attribute sharing rides along as its own field per pair.

**Active metric: cosine similarity** (`nomic-embed-text` embeddings over each grounded
concept's SNOMED preferred name) -- consistent with Stage 5, which also uses cosine
throughout. Both ancestor-set Jaccard similarity and Wu-Palmer (graph-structural, is-a
hierarchy) are fully implemented below but commented out -- see the note at the top of
this notebook for why, and for the not-yet-built causal-chain idea. Note that under
cosine there's no equivalent to the LCS "via: X" explanation the graph-based metrics
gave -- cosine doesn't go through an intermediate shared-ancestor concept, so
`lcs_sctid`/`lcs_name`/`lcs_depth` are `None` in the active path.

**Duplicate-concept exclusion**: if two extracted terms ground to the *same* SCTID (e.g.
"cough" vs "coughing" -- two extractions of one underlying symptom), cosine similarity
between identical text is exactly 1.0 -- the same problem the graph-based metrics had,
for the same underlying reason. Left alone, that artificial 1.0 can dominate the ranked
list and swallow the big-gap cutoff, hiding real significant pairs beneath it. Such
pairs are still reported in the output (`duplicate_concept: true`), just excluded from
the cutoff search itself.

**Group clustering**: significant pairs are grouped into clusters via connected
components (nodes = grounded terms, edges = significant pairs), so a chain of pairwise
connections (A-B significant, B-C significant) surfaces as one 3-symptom cluster rather
than two separate, seemingly-unrelated pairs. Every grounded term appears in exactly one
cluster -- terms with no significant connection to anything else form their own
singleton cluster.


In [4]:
def load_grounded_symptoms(routed_terms: dict) -> dict:
    """{term: sctid} for every CURRENT_SYMPTOMS term Stage 5 successfully grounded."""
    grounded = {}
    for branch in routed_terms.get("routed_branches", []):
        for s in branch.get("symptoms", []):
            routing = s.get("routing")
            if routing and routing.get("grounded"):
                grounded[s["term"]] = routing["grounded"]["sctid"]
    return grounded


def find_clusters(terms: list, significant_pairs: list) -> list:
    """Connected components over the graph of significant pairs (nodes = grounded
    terms, edges = pairs flagged `significant_connection`). Groups symptoms that are
    only connected through a chain (A-B significant, B-C significant, but A-C never
    scored as significant) into one cluster, rather than reporting them as two
    separate, seemingly-unrelated pairs. A term with no significant connection to
    anything else forms its own singleton cluster, so every grounded term appears
    in exactly one cluster."""
    parent = {t: t for t in terms}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    for p in significant_pairs:
        union(p["term_a"], p["term_b"])

    groups = {}
    for t in terms:
        groups.setdefault(find(t), []).append(t)

    clusters = sorted(groups.values(), key=len, reverse=True)
    return [{"terms": sorted(c), "size": len(c)} for c in clusters]


def compute_symptom_relations(grounded: dict) -> dict:
    """All pairwise cosine-similarity scores among grounded symptoms' SNOMED names,
    plus each pair's attribute-sharing score, with the admission's own big-gap cutoff
    applied to the sorted cosine scores. Attribute sharing rides along as its own
    field per pair. Significant pairs are then grouped into clusters via connected
    components (find_clusters above), so multi-symptom groups are surfaced directly
    instead of just a flat list of pairs.

    Pairs where both terms grounded to the *same* SCTID are flagged `duplicate_concept`
    and excluded from the cutoff search -- see the markdown cell above for why."""
    terms = list(grounded.keys())
    names = {t: get_concept_name(grounded[t]) for t in terms}

    # -- ACTIVE: cosine similarity (text-embedding, nomic-embed-text) ---------------
    vecs = {t: embed(names[t]) for t in terms}

    pairs = []
    for i in range(len(terms)):
        for j in range(i + 1, len(terms)):
            term_a, term_b = terms[i], terms[j]
            score = round(cosine_sim(vecs[term_a], vecs[term_b]), 4)
            lcs_sctid = None   # cosine has no shared-ancestor concept to report
            lcs_name = None
            lcs_depth = None

            # -- COMMENTED OUT -- ancestor-set Jaccard similarity (is-a hierarchy) -----
            # Considers the whole shared lineage rather than one deepest ancestor, but
            # produced a tie-heavy score range that broke biggest_gap_cutoff()'s
            # median/MAD statistic on real data (see the note at the top of this
            # notebook) -- needs cutoff recalibration before being trustworthy.
            # jac_result = ancestor_jaccard_with_lcs(grounded[term_a], grounded[term_b])
            # if jac_result is None:
            #     continue
            # score = jac_result["jaccard"]
            # lcs_sctid = jac_result["lcs_sctid"]
            # lcs_name = get_concept_name(lcs_sctid) if lcs_sctid else None
            # lcs_depth = jac_result["lcs_depth"]

            # -- COMMENTED OUT -- Wu-Palmer (single deepest shared ancestor only) -------
            # Uncomment to test against real data now that the duplicate-concept
            # exclusion and grounding fixes are in place.
            # wp_result = wu_palmer_with_lcs(grounded[term_a], grounded[term_b])
            # if wp_result is None:
            #     continue
            # score = wp_result["score"]
            # lcs_sctid = wp_result["lcs_sctid"]
            # lcs_name = get_concept_name(lcs_sctid)
            # lcs_depth = wp_result["lcs_depth"]

            # -- FUTURE WORK, NOT IMPLEMENTED -- multi-hop causal-chain traversal ------
            # A third alternative worth testing later: instead of is-a graph structure
            # (Jaccard/Wu-Palmer above) or textual similarity (cosine above), check
            # whether a chain of due_to/cause_of relations connects the two concepts
            # (e.g. HCC -> due_to -> liver failure -> due_to -> hepatic encephalopathy).
            # This would capture genuine causal/mechanistic convergence that neither
            # taxonomic distance nor name similarity can see. Would need a BFS over
            # get_causal_targets() run outward from both concepts, plus a decision on
            # max chain length -- not built here yet.

            attr_score, attr_details = compute_attribute_score(grounded[term_a], grounded[term_b])
            pairs.append({
                "term_a": term_a,
                "term_b": term_b,
                "score": score,
                "lcs_sctid": lcs_sctid,
                "lcs_name": lcs_name,
                "lcs_depth": lcs_depth,
                "attribute_score": attr_score,
                "shared_attributes": attr_details,
                "duplicate_concept": grounded[term_a] == grounded[term_b],
            })

    pairs.sort(key=lambda p: p["score"], reverse=True)

    # Duplicates don't compete for the cutoff -- they're the same concept counted
    # twice (a duplicate extraction), not two distinct concepts that happen to be
    # highly related, and their guaranteed 1.0 score would otherwise dominate it.
    rankable = [p for p in pairs if not p["duplicate_concept"]]
    scored = [(f"{p['term_a']}|{p['term_b']}", p["score"]) for p in rankable]
    cutoff_idx, gap, is_significant = biggest_gap_cutoff(scored)
    n_kept = (cutoff_idx + 1) if cutoff_idx is not None else len(rankable)
    significant_ids = {id(p) for p in rankable[:n_kept]} if is_significant else set()

    for p in pairs:
        p["significant_connection"] = id(p) in significant_ids

    significant_pairs = [p for p in pairs if p["significant_connection"]]
    clusters = find_clusters(terms, significant_pairs)

    return {
        "n_grounded_symptoms": len(terms),
        "n_pairs": len(pairs),
        "n_duplicate_concept_pairs": sum(p["duplicate_concept"] for p in pairs),
        "cutoff": {"gap": gap, "is_significant": is_significant, "n_kept": n_kept},
        "pairs": pairs,
        "clusters": clusters,
    }


## 4. Test on one real admission before running the full batch

Uses a patient with many grounded symptoms (richer pairwise data), so the big-gap cutoff has
enough pairs to be statistically meaningful.


In [5]:
test_path = RECORDS_DIR / "patient_17774110" / "admissions"
test_adm = sorted(test_path.iterdir())[0]
with open(test_adm / STAGE_05_OUTPUT / "routed_terms.json", encoding="utf-8") as f:
    test_routed = json.load(f)

test_grounded = load_grounded_symptoms(test_routed)
print(f"Grounded symptoms: {list(test_grounded.keys())}")

test_result = compute_symptom_relations(test_grounded)
print(f"\n{test_result['n_pairs']} pairs ({test_result['n_duplicate_concept_pairs']} duplicate-concept), cutoff: {test_result['cutoff']}\n")
for p in test_result["pairs"]:
    flag = " <-- significant" if p["significant_connection"] else ""
    attr_note = f"  [attr: {p['attribute_score']:.2f}]" if p["attribute_score"] > 0 else ""
    dup_note = "  [DUPLICATE CONCEPT -- excluded from cutoff]" if p["duplicate_concept"] else ""
    print(f"{p['score']:.4f}  {p['term_a']!r} <-> {p['term_b']!r}{attr_note}{dup_note}{flag}")

print(f"\nClusters ({len(test_result['clusters'])}):")
for c in test_result["clusters"]:
    print(f"  size {c['size']}: {c['terms']}")


Grounded symptoms: ['Septic shock', 'Hypoxic respiratory failure', 'Epistaxis', 'GI bleeding', 'Liver failure', 'Renal failure', 'Hepatic encephalopathy', 'Multifocal metastatic HCC', 'Atelectasis', 'Thrombocytopenia', 'Atrial fibrillation/flutter']

55 pairs, cutoff: {'gap': 0.045499999999999985, 'is_significant': True, 'n_kept': 6}

0.6000  'Septic shock' <-> 'Liver failure'  (via: Acute disease) <-- significant
0.6000  'Hypoxic respiratory failure' <-> 'Hepatic encephalopathy'  (via: Metabolic disease) <-- significant
0.6000  'GI bleeding' <-> 'Liver failure'  (via: Disorder of digestive system) <-- significant
0.5556  'Liver failure' <-> 'Renal failure'  (via: Disorder of abdomen) <-- significant
0.5556  'Atelectasis' <-> 'Atrial fibrillation/flutter'  (via: Disorder of thorax) <-- significant
0.5455  'Septic shock' <-> 'Renal failure'  (via: Acute disease) <-- significant
0.5000  'Epistaxis' <-> 'GI bleeding'  (via: Bleeding)  [attr: 0.15]
0.4615  'GI bleeding' <-> 'Renal failure'

## 5. Run across all patients

Only admissions with at least 2 grounded symptoms produce any pairs. Nothing here re-grounds or
re-explores neighborhoods — this is pure graph distance and attribute lookups between concepts
Stage 5 already resolved, so no LLM calls and no new grounding/embedding calls, just is-a and
relations traversal.


In [6]:
processed_admissions = 0
processed_pairs = 0

for patient_dir in patients:
    patient_id = patient_dir.name.replace("patient_", "")
    adm_root = patient_dir / "admissions"
    adm_dirs = sorted(adm_root.iterdir()) if adm_root.exists() else []

    for adm_dir in adm_dirs:
        routed_path = adm_dir / STAGE_05_OUTPUT / "routed_terms.json"
        if not routed_path.exists():
            print(f"  SKIP {patient_id}/{adm_dir.name} -- no Stage 5 output yet")
            continue

        with open(routed_path, encoding="utf-8") as f:
            routed = json.load(f)

        grounded = load_grounded_symptoms(routed)
        if len(grounded) < 2:
            print(f"Patient {patient_id} | {adm_dir.name} | {len(grounded)} grounded symptom(s) -- not enough for pairs")
            continue

        result = compute_symptom_relations(grounded)
        result["patient_id"] = routed.get("patient_id")
        result["admission_id"] = routed.get("admission_id")
        result["attribute_weights"] = {
            "cause_of": WEIGHT_CAUSE_OF,
            "causative_agent": WEIGHT_CAUSATIVE_AGENT,
            "finding_site": WEIGHT_FINDING_SITE,
            "associated_morphology": WEIGHT_ASSOCIATED_MORPHOLOGY,
            "pathological_process": WEIGHT_PATHOLOGICAL_PROCESS,
        }

        out_dir = adm_dir / STAGE_06_OUTPUT
        out_dir.mkdir(exist_ok=True)
        with open(out_dir / "symptom_relations.json", "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2)

        processed_admissions += 1
        processed_pairs += result["n_pairs"]
        n_sig = sum(1 for p in result["pairs"] if p["significant_connection"])
        n_attr = sum(1 for p in result["pairs"] if p["attribute_score"] > 0)
        n_dup = result["n_duplicate_concept_pairs"]
        n_clusters = len(result["clusters"])
        print(f"Patient {patient_id} | {adm_dir.name} | {result['n_pairs']} pairs, {n_sig} significant, {n_attr} with shared attributes, {n_dup} duplicate-concept, {n_clusters} clusters")

print(f"\nDone. {processed_admissions} admissions processed, {processed_pairs} pairs computed.")


Patient 10361982 | hadm_24286431 | 3 pairs, 0 significant, 1 with shared attributes
Patient 10426859 | hadm_29908281 | 10 pairs, 3 significant, 0 with shared attributes
Patient 10458324 | hadm_21744342 | 3 pairs, 0 significant, 1 with shared attributes
Patient 11251337 | hadm_29568708 | 3 pairs, 0 significant, 0 with shared attributes
Patient 11474876 | hadm_29672491 | 1 pairs, 0 significant, 0 with shared attributes
Patient 11607177 | hadm_23293838 | 1 pairs, 0 significant, 0 with shared attributes
Patient 12007928 | hadm_23749816 | 10 pairs, 1 significant, 0 with shared attributes
Patient 13196707 | hadm_21475988 | 15 pairs, 2 significant, 0 with shared attributes
Patient 13508515 | hadm_21834271 | 1 grounded symptom(s) -- not enough for pairs
Patient 13952483 | hadm_23852410 | 78 pairs, 1 significant, 1 with shared attributes
Patient 16014068 | hadm_29042843 | 6 pairs, 1 significant, 0 with shared attributes
Patient 17774110 | hadm_27339772 | 55 pairs, 6 significant, 1 with shared a

## 6. Inspect one admission

In [8]:
EXAMPLE_IDX = 9
example_patient = patients[EXAMPLE_IDX]
adm_dirs = sorted((example_patient / "admissions").iterdir())
relations_path = adm_dirs[0] / STAGE_06_OUTPUT / "symptom_relations.json"

if not relations_path.exists():
    print(f"No Stage 6 output for {example_patient.name} (fewer than 2 grounded symptoms).")
else:
    with open(relations_path, encoding="utf-8") as f:
        example = json.load(f)

    print(f'Patient {example["patient_id"]} | Admission {example["admission_id"]}')
    print(f'{example["n_grounded_symptoms"]} grounded symptoms, {example["n_pairs"]} pairs')
    print(f'Cutoff: {example["cutoff"]}')
    print(f'Attribute weights used: {example.get("attribute_weights")}')
    print()
    for p in example["pairs"]:
        flag = " <-- significant" if p["significant_connection"] else ""
        attr_note = f"  [attr: {p['attribute_score']:.2f}, shared: {p['shared_attributes']}]" if p["attribute_score"] > 0 else ""
        dup_note = "  [DUPLICATE CONCEPT]" if p.get("duplicate_concept") else ""
        print(f'  {p["score"]:.4f}  {p["term_a"]!r} <-> {p["term_b"]!r}{attr_note}{dup_note}{flag}')

    print()
    print(f"Clusters ({len(example.get('clusters', []))}):")
    for c in example.get("clusters", []):
        print(f"  size {c['size']}: {c['terms']}")


Patient 13952483 | Admission 23852410
13 grounded symptoms, 78 pairs
Cutoff: {'gap': 0.2632, 'is_significant': True, 'n_kept': 1}
Attribute weights used: {'cause_of': 0.45, 'causative_agent': 0.4, 'finding_site': 0.25, 'associated_morphology': 0.15, 'pathological_process': 0.15}

  1.0000  'cough' <-> 'coughing'  (via: Cough) <-- significant
  0.7368  'neutropenia' <-> 'thrombocytopenia'  (via: Cytopenia)
  0.7143  'odynophagia' <-> 'chest pain'  (via: Pain finding at anatomical site)
  0.5000  'hypotension' <-> 'pain with swallowing'  (via: Disorder of body system)
  0.5000  'odynophagia' <-> 'nausea'  (via: Digestive system finding)
  0.4444  'cough' <-> 'malaise'  (via: Functional finding)
  0.4444  'cough' <-> 'thrombocytopenia'  (via: Functional finding)
  0.4444  'malaise' <-> 'coughing'  (via: Functional finding)
  0.4444  'thrombocytopenia' <-> 'coughing'  (via: Functional finding)
  0.4000  'hypotension' <-> 'thrombocytopenia'  (via: Disease)
  0.4000  'odynophagia' <-> 'cough